# Chapter 08: Retrieval-Augmented Generation (RAG)

RAG fixes the main weakness of a fixed LLM: **it doesn't know facts added after training**. Instead of retraining, we:

1. Embed a knowledge base of documents
2. At query time, retrieve the most relevant chunks
3. Feed them into the prompt as context

```
Query → embed → vector search → top-k chunks → [chunks + query] → LLM → answer
```

## Setup

In [ ]:
# !pip install sentence-transformers scikit-learn transformers torch
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
print('Ready')

## Step 1 — Build a knowledge base

In [ ]:
# Small but realistic knowledge base
knowledge_base = [
    "The Eiffel Tower was completed in 1889 and stands 330 metres tall.",
    "Python was created by Guido van Rossum and first released in 1991.",
    "The mitochondria is often called the powerhouse of the cell.",
    "Transformer models use self-attention to process sequences in parallel.",
    "The speed of light in a vacuum is approximately 299,792 km/s.",
    "BERT stands for Bidirectional Encoder Representations from Transformers.",
    "GPT uses a decoder-only transformer architecture for text generation.",
    "Fine-tuning adapts a pre-trained model to a specific task with a small dataset.",
    "Embeddings map discrete tokens to continuous vector spaces.",
    "The French Revolution began in 1789 and ended in the late 1790s.",
]
print(f"Knowledge base: {len(knowledge_base)} chunks")

## Step 2 — Embed the knowledge base

In [ ]:
embedder = SentenceTransformer("all-MiniLM-L6-v2")
kb_embeddings = embedder.encode(knowledge_base)
print("KB embeddings shape:", kb_embeddings.shape)

## Step 3 — Retrieval function

In [ ]:
def retrieve(query, top_k=3):
    q_emb = embedder.encode([query])
    scores = cosine_similarity(q_emb, kb_embeddings)[0]
    top_idx = scores.argsort()[::-1][:top_k]
    return [(knowledge_base[i], round(float(scores[i]), 3)) for i in top_idx]

# Test retrieval
results = retrieve("What year was the Eiffel Tower built?")
for chunk, score in results:
    print(f"  [{score}] {chunk}")

## Step 4 — Generate an answer using retrieved context

In [ ]:
from transformers import pipeline

qa = pipeline("text2text-generation", model="google/flan-t5-small")

def rag_answer(question):
    chunks = retrieve(question, top_k=2)
    context = " ".join(c for c,_ in chunks)
    prompt = f"Answer the question based on the context.\nContext: {context}\nQuestion: {question}\nAnswer:"
    return qa(prompt, max_new_tokens=60)[0]["generated_text"]

questions = [
    "What year was Python first released?",
    "What does BERT stand for?",
    "How fast does light travel?",
]
for q in questions:
    print(f"Q: {q}")
    print(f"A: {rag_answer(q)}")
    print()

## Step 5 — What happens without RAG?

In [ ]:
gen = pipeline("text-generation", model="distilgpt2", max_new_tokens=40, do_sample=False)

q = "What does BERT stand for?"
no_rag = gen(f"Q: {q} A:")[0]["generated_text"]
with_rag = rag_answer(q)

print("Without RAG:", no_rag)
print()
print("With RAG:   ", with_rag)

## Summary

| Step | Tool |
|------|------|
| Embed chunks | `sentence-transformers` |
| Retrieve | cosine similarity |
| Generate | any LLM (here: flan-t5-small) |

In production, replace the numpy similarity search with a vector database (FAISS, Chroma, Qdrant) for millions of documents.